# Panama elektrikoormuse uuriv andmeanalüüs

Andmekvaliteeti kontrollitakse kogu lähtefailil; tunnusevalikut toetav statistika ja kõik alljärgnevad joonised kasutavad ainult 2015–2017 treeningandmeid. Korrelatsioon ei ole põhjuslik seos. Arvud loetakse tegeliku käivituse tulemustest, tulemusi ei ole notebooki välja mõeldud.

In [ ]:
from pathlib import Path
from functools import partial
import sys
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "run_pipeline.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.common import latest_run, read_json
from src.reporting import show_table, show_figure, show_run

RUN = latest_run()
table = partial(show_table, RUN, limit=30)
figure = partial(show_figure, RUN)
show_run(RUN)

## Andmete struktuur ja kvaliteet

Kvaliteediaudit kirjeldab ka vigu; toru peatub, kui puuduvad väärtused või tunnid takistavad korrektset modelleerimist. Ebatavalised koormused säilitatakse.

In [ ]:
quality = read_json(RUN / "eda/quality.json")
display(
    Markdown(
        f"Kontrolliti {quality['rows']} vaatlust ja {quality['columns']} veergu. Puuduvaid tunde: {quality['missing_hours']}; duplikaatseid ajatempleid: {quality['duplicate_timestamps']}; alla 200 ühiku koormusi: {quality['load_below_200']}. Koormuse miinimumi käsitletakse uuritava vaatlusena, sest eemaldamiseks puudub tõend."
    )
)
table(
    "eda/train_summary.csv",
    rename={
        "count": "Vaatlusi",
        "mean": "Keskmine",
        "std": "Standardhälve",
        "min": "Miinimum",
        "25%": "25. protsentiil",
        "50%": "Mediaan",
        "75%": "75. protsentiil",
        "max": "Maksimum",
    },
)
display(
    Markdown(
        "Tabel kirjeldab ainult treeningperioodi jaotusi algsetes CSV ühikutes. Mõõtühikute vastuolud on päritoludokumendis; hajuvus ei erista mõõtmisviga tegelikust sündmusest. Tunnused kontrollitakse valideerimisaastal."
    )
)

In [ ]:
captions = read_json(RUN / "eda/interpretations.json")
for filename, text in captions.items():
    figure("figures/eda/" + filename, text)